# Paper 17: Point-in-time universe effect

Measure how much of the v1 path came from survivorship (today's S&P 500 list) versus the point-in-time members in v2 baseline. Current-list is the leak, not a live candidate — keep PIT even if it looks better.

Scored against **v2 baseline**. Kill rules are frozen in `Research/book-construction-status.md`:

- Train 2022 calendar more than 5pp worse than SPUS
- Train max drawdown worse than SPUS
- Train Calmar below both v2 baseline and SPUS
- Test 2023 calendar more than 15pp worse than v2 baseline

**Variants.** v2-baseline (PIT) vs current-list (pit=false).


In [2]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "monterey" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(f"Monterey repo root not found from {start}")

_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
try:
    import monterey  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{_ROOT}[research]"])

from monterey.diagnostics.performance import yearly
from monterey.research import boot, compare
from monterey.sprints import (
    run_sprint,
    score_against_baseline,
    survivorship_in_book,
    universe_survivorship,
    v2_snapshot_hash,
)

rd = boot("paper-17-pit-universe", snapshot=v2_snapshot_hash())
print("snapshot", rd.manifest.get("hash"))
census = universe_survivorship(rd)
print(census.agg({"n_pit": "mean", "n_current": "mean", "joiners": "mean", "leavers": "mean"}))
census.tail()


        field                   value
       window 2020-01-02 → 2026-09-30
     universe    sp500, point-in-time
      symbols                     624
    snapshots                      83
   price rows                 1126163
    dividends                    9564
filing events                   59178
   halalquant                   0.4.0
snapshot hash            bfb52f463228
snapshot bfb52f463228
n_pit       503.7037
n_current   503.0000
joiners      55.6296
leavers      56.3333
dtype: float64


,date,n_pit,n_current,joiners,leavers
76,2026-05-29,503,503,10,10
77,2026-06-30,503,503,6,6
78,2026-07-31,503,503,6,6
79,2026-08-31,503,503,0,0
80,2026-09-25,503,503,0,0


In [3]:
ledgers, table = run_sprint("17", rd)
table


universe census (month-end mean): PIT 504 vs current 503; joiners 56, leavers 56
current-list book held 39 joiner names that were not in the index that day
PIT book held 16 leaver names missing from today's list
                     total_return   cagr  volatility  max_drawdown  calmar  sharpe  sortino  turnover  costs_pct  dividends_pct  avg_exposure  switches          hash
book                                                                                                                                                                 
fcf-sma-v2-baseline        1.0464 0.1120      0.1462       -0.2482  0.4513  0.6653   0.8464    4.1570     0.0561         0.0560        0.8035   36.0000  6ca86844209e
current-list               1.1160 0.1178      0.1473       -0.2459  0.4792  0.6966   0.8858    4.1828     0.0563         0.0556        0.8032   36.0000  c7cda04e91e6
SPUS (total return)        2.1167 0.1836      0.2114       -0.3080  0.5961  0.8114   1.0810       NaN        NaN            

,total_return,cagr,volatility,max_drawdown,calmar,sharpe,sortino,turnover,costs_pct,dividends_pct,avg_exposure,switches,hash
book,,,,,,,,,,,,,
fcf-sma-v2-baseline,1.0464,0.1120,0.1462,-0.2482,0.4513,0.6653,0.8464,4.1570,0.0561,0.0560,0.8035,36.0000,6ca86844209e
current-list,1.1160,0.1178,0.1473,-0.2459,0.4792,0.6966,0.8858,4.1828,0.0563,0.0556,0.8032,36.0000,c7cda04e91e6
SPUS (total return),2.1167,0.1836,0.2114,-0.3080,0.5961,0.8114,1.0810,NaN,NaN,NaN,NaN,NaN,NaN
SPY (total return),1.6113,0.1530,0.2005,-0.3372,0.4537,0.7132,0.8841,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
baseline, current = ledgers
ghosts = survivorship_in_book(current, rd)
leavers = survivorship_in_book(baseline, rd)
joiners = ghosts[ghosts["kind"] == "joiner"] if not ghosts.empty else ghosts
gone = leavers[leavers["kind"] == "leaver"] if not leavers.empty else leavers
print("current-list names held before they joined the index")
print(joiners.groupby("symbol")["weight"].agg(["count", "mean", "max"]).sort_values("max", ascending=False).head(20) if not joiners.empty else joiners)
print("PIT names held after they left today's list")
print(gone.groupby("symbol")["weight"].agg(["count", "mean", "max"]).sort_values("max", ascending=False).head(20) if not gone.empty else gone)


current-list names held before they joined the index
        count   mean    max
symbol                     
ABNB      299 0.0081 0.0110
MRVL      270 0.0041 0.0098
MRNA       34 0.0077 0.0095
COIN      309 0.0034 0.0073
VEEV      318 0.0053 0.0071
DASH      416 0.0039 0.0069
KDP        45 0.0062 0.0068
NXPI       79 0.0051 0.0067
DDOG      830 0.0032 0.0056
PLTR      453 0.0037 0.0053
WDAY      141 0.0040 0.0050
VMRK       34 0.0046 0.0048
KVUE       38 0.0035 0.0039
CRH        44 0.0035 0.0038
TYL        18 0.0026 0.0029
CSGP       50 0.0026 0.0028
RDDT      499 0.0014 0.0024
FERG      127 0.0020 0.0023
APP       254 0.0013 0.0023
MPWR      113 0.0019 0.0022
PIT names held after they left today's list
        count   mean    max
symbol                     
ENPH      673 0.0020 0.0046
PAYC     1220 0.0015 0.0034
ETSY      199 0.0020 0.0033
MKTX      340 0.0024 0.0032
QRVO      581 0.0016 0.0027
BLDR      550 0.0011 0.0023
TTD       251 0.0009 0.0022
LW        312 0.0012 0.0021
BIO    

In [5]:
import matplotlib.pyplot as plt

figures = Path("figures")
figures.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(census["date"], census["n_pit"], label="PIT members")
axes[0].plot(census["date"], census["n_current"], label="today's list")
axes[0].set_title("S&P 500 membership: point-in-time vs end-date list")
axes[0].legend()
for book in ledgers:
    nav = book.nav_series()
    axes[1].plot(nav.index, nav / nav.iloc[0], label=book.spec.id)
spus = rd.total_return("SPUS").reindex(ledgers[0].nav_series().index).ffill()
axes[1].plot(spus.index, spus / spus.iloc[0], label="SPUS", alpha=0.7)
axes[1].set_title("NAV, start = 1")
axes[1].legend()
fig.tight_layout()
fig.savefig(figures / "pit-vs-current.png")
plt.close(fig)
print("wrote", figures / "pit-vs-current.png")
print(yearly(ledgers[0].nav_series()).join(yearly(ledgers[1].nav_series()), lsuffix="_pit", rsuffix="_current"))


wrote figures/pit-vs-current.png
      return_pit  intra_year_drawdown_pit   year_end_pit  return_current  intra_year_drawdown_current  year_end_current
year                                                                                                                   
2020      0.0528                  -0.1909 1,051,740.3783          0.0495                      -0.1912    1,048,408.3189
2021      0.2831                  -0.0809 1,349,537.5092          0.2892                      -0.0807    1,351,598.7934
2022     -0.2080                  -0.2071 1,068,786.3906         -0.2069                      -0.2062    1,071,969.0450
2023      0.1525                  -0.0802 1,231,734.5296          0.1603                      -0.0782    1,243,787.0389
2024      0.2741                  -0.0723 1,569,404.1430          0.2869                      -0.0756    1,600,679.4163
2025      0.1250                  -0.1350 1,765,558.5937          0.1283                      -0.1383    1,806,102.5686
2026   

In [6]:
verdict = score_against_baseline(current, baseline, rd)
print(current.spec.id, "KILLED" if verdict["killed"] else "looks fine on kill bars", "; ".join(verdict["reasons"]) or "pass")
print("Architecture: keep universe.pit=true. current-list is survivorship, not a live rule.")


current-list looks fine on kill bars pass
Architecture: keep universe.pit=true. current-list is survivorship, not a live rule.


## Conclusion

This sprint never turned off Halal screens. Both books still drop banned activities and AAOIFI debt / cash / receivables. A bank or a name over the 30% debt line is out in **both** runs.

The only thing that changed is **who is allowed to sit in that Halal funnel**.

- **Point-in-time (v2, the live rule).** On 15 June 2020 the universe is tickers that were in the S&P 500 **on 15 June 2020**. Then Halal-screen. Then FCF-rank.
- **Today’s list (`pit: false`).** “Today” is the snapshot end date, **30 September 2026** — a frozen list of ~503 names. On 15 June 2020 the backtest still uses **that 2026 list**, then Halal-screens it.

So “today’s list in 2020” means: pretend the 2026 S&P members were already the S&P in 2020.

**What survivorship is.** The S&P 500 is not the same 500 companies forever.

- **Joiners.** Palantir is on the 2026 list. It was **not** in the index in 2020. The leaked book can still own PLTR in 2020 if it passes Halal. The honest book cannot. Same for Airbnb, Datadog, and the other 39 joiners we held.
- **Leavers.** A name that *was* in the S&P in 2020 but later left is **missing** from the 2026 list. The leaked book never holds it in 2020 either. You only see companies that “survived” until 2026. The PIT book still held 16 of those leavers.

**The number.** Honest v2: **+105%** total, **11.2%** CAGR, **−24.8%** max drawdown, Calmar **0.45**. Today’s list: **+112%**, **11.8%** CAGR, **−24.6%** DD, Calmar **0.48**. Kill bars **pass**. That is the cheat looking like a better book. The extra ~0.6pp CAGR is index membership look-ahead, mostly in 2023–26. 2020–22 barely move.

**Architecture.** Keep `universe.pit: true`. Do not promote `pit: false`. A Halal fund does not get to keep a name because the backtest likes it — whether that name was not in the index yet, or not Halal that day. v2 already uses the honest membership list. This paper only put a number on the cheat.